# 🚗 Notebook 01 — Установка окружения и загрузка данных

**Дипломная работа:** Система автономного управления автомобилем (Behavioral Cloning)

## Что делаем в этом ноутбуке:
1. Устанавливаем зависимости
2. Подключаем Google Drive
3. Скачиваем датасет Udacity с Kaggle
4. Изучаем структуру данных
5. Строим гистограмму распределения углов руля

---
> ⚠️ **Перед запуском:** убедись что выбрана среда с GPU:  
> `Среда выполнения → Сменить тип среды → GPU (T4)`

## Шаг 1. Установка зависимостей

In [ ]:
# Устанавливаем все необходимые библиотеки
# (в Colab большинство уже есть, но версии могут отличаться)
!pip install -q torch torchvision numpy pandas matplotlib opencv-python \
                 scikit-learn ultralytics gdown tqdm seaborn Pillow

print('✓ Зависимости установлены')

## Шаг 2. Подключение Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Путь к папке проекта на Google Drive
# Измени если твоя папка называется иначе
PROJECT_DIR = '/content/drive/MyDrive/diploma'
DATA_DIR    = os.path.join(PROJECT_DIR, 'data')
MODELS_DIR  = os.path.join(PROJECT_DIR, 'models')
OUTPUTS_DIR = os.path.join(PROJECT_DIR, 'outputs')

# Создаём папки если не существуют
for d in [DATA_DIR, MODELS_DIR, OUTPUTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'✓ Google Drive подключён')
print(f'  Проект:  {PROJECT_DIR}')
print(f'  Данные:  {DATA_DIR}')
print(f'  Модели:  {MODELS_DIR}')

## Шаг 3. Загрузка датасета Udacity

### Откуда брать данные?

**Вариант A (рекомендуется): Kaggle**
1. Зарегистрируйся на [kaggle.com](https://www.kaggle.com)
2. Скачай датасет: `ronamgir/udacity-car-dataset-simulator-challenge`
3. Загрузи архив в папку `diploma/data/` на Google Drive

**Вариант B: Прямая ссылка (если Kaggle недоступен)**  
Используем датасет от Udacity (симулятор, около 300 MB)

In [ ]:
import zipfile

# ── Вариант A: Kaggle API ──────────────────────────────────
# Если у тебя есть kaggle.json (токен), раскомментируй:

# from google.colab import files
# files.upload()  # загрузи kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d ronamgir/udacity-car-dataset-simulator-challenge -p {DATA_DIR}
# with zipfile.ZipFile(f'{DATA_DIR}/udacity-car-dataset-simulator-challenge.zip') as z:
#     z.extractall(DATA_DIR)

# ── Вариант B: gdown (если файл уже на Drive) ─────────────
# Если уже загрузил архив вручную, просто распакуй:

zip_path = os.path.join(DATA_DIR, 'dataset.zip')
if os.path.exists(zip_path):
    print('Распаковываем архив...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATA_DIR)
    print('✓ Данные распакованы')
else:
    print('⚠ Файл dataset.zip не найден в', DATA_DIR)
    print('  Загрузи архив с Kaggle и помести его в:', DATA_DIR)

# Смотрим что появилось
print('\nСодержимое папки данных:')
for item in os.listdir(DATA_DIR):
    print(f'  {item}')

## Шаг 4. Изучение структуры данных

In [ ]:
import pandas as pd
import numpy as np

# Ищем файл driving_log.csv
csv_path = None
for root, dirs, files in os.walk(DATA_DIR):
    for f in files:
        if f == 'driving_log.csv':
            csv_path = os.path.join(root, f)
            break

if csv_path is None:
    raise FileNotFoundError('driving_log.csv не найден! Проверь распаковку архива.')

print(f'✓ Найден CSV: {csv_path}')

# Загружаем CSV
df = pd.read_csv(csv_path, header=None,
                 names=['center', 'left', 'right', 'steering', 'throttle', 'reverse', 'speed'])

print(f'\nРазмер датасета: {df.shape[0]} строк × {df.shape[1]} столбцов')
print('\nПервые 5 строк:')
df.head()

In [ ]:
# Статистика по числовым колонкам
print('Статистика по данным:')
df[['steering', 'throttle', 'speed']].describe().round(4)

In [ ]:
# Проверяем пропущенные значения
missing = df.isnull().sum()
print('Пропущенные значения:')
print(missing)
print(f'\nИтого строк с пропусками: {df.isnull().any(axis=1).sum()}')

## Шаг 5. Визуализация распределения углов руля

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# ── Гистограмма углов ──
ax = axes[0]
ax.hist(df['steering'], bins=50, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Прямая езда (0)')
ax.set_xlabel('Угол руля', fontsize=12)
ax.set_ylabel('Количество кадров', fontsize=12)
ax.set_title('Распределение углов руля (до балансировки)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

straight = (df['steering'].abs() < 0.05).mean() * 100
ax.text(0.98, 0.95, f'Прямая езда: {straight:.1f}%',
        transform=ax.transAxes, ha='right', va='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# ── Скорость во времени ──
ax = axes[1]
ax.plot(df['speed'].values[:500], color='green', linewidth=1.2)
ax.set_xlabel('Кадр', fontsize=12)
ax.set_ylabel('Скорость', fontsize=12)
ax.set_title('Скорость автомобиля (первые 500 кадров)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'data_analysis.png'), dpi=150)
plt.show()
print(f'\n⚠ Замечание: {straight:.1f}% данных — прямая езда.')
print('  Это типичный дисбаланс. Решим его в notebook 02 (балансировка).')

## Шаг 6. Просмотр примеров изображений

In [ ]:
import cv2
from IPython.display import display

# Папка с изображениями
img_dir = os.path.join(os.path.dirname(csv_path), 'IMG')
print(f'Папка изображений: {img_dir}')

# Показываем 6 случайных изображений из центральной камеры
sample_rows = df.sample(6, random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(14, 6))
fig.suptitle('Примеры изображений с центральной камеры', fontsize=13, fontweight='bold')

for ax, (_, row) in zip(axes.flatten(), sample_rows.iterrows()):
    img_path = os.path.join(img_dir, os.path.basename(str(row['center']).strip()))
    if os.path.exists(img_path):
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f'Угол руля: {row["steering"]:.3f}', fontsize=10)
    else:
        ax.text(0.5, 0.5, 'Файл не найден', ha='center', va='center')
    ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'sample_images.png'), dpi=150)
plt.show()

## Итог

✅ Окружение настроено  
✅ Google Drive подключён  
✅ Датасет загружен и изучен  
✅ Выявлен дисбаланс: большинство кадров — прямая езда  

**Следующий шаг:** `02_preprocessing.ipynb` — предобработка и балансировка датасета

In [ ]:
# Сохраняем пути для следующих ноутбуков
import json
config = {
    'project_dir': PROJECT_DIR,
    'data_dir':    DATA_DIR,
    'models_dir':  MODELS_DIR,
    'outputs_dir': OUTPUTS_DIR,
    'csv_path':    csv_path,
    'img_dir':     img_dir,
}
config_path = os.path.join(PROJECT_DIR, 'config.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f'✓ Конфигурация сохранена: {config_path}')